# Coupled climate–photochemistry model of an Archean Earth around an M dwarf (GJ 876)

This is the same experiment as `archean_photochem_climate.ipynb`, with one change: the star. The Sun at 3 Ga is replaced by the M dwarf **GJ 876** (T<sub>eff</sub> ≈ 3300 K), using its observed [MUSCLES](https://archive.stsci.edu/prepds/muscles/) spectrum.

To isolate the effect of the *spectrum*, the planet is placed at the orbital distance where GJ 876 delivers the **same total instellation** as the 3 Ga Sun gave Earth. Planet, atmosphere, surface fluxes, albedo, humidity and the coupling loop are all identical to the first notebook.

What changes physically:

- **Less UV, more red/near-IR.** An M dwarf emits far less near-UV/visible light than the Sun at equal bolometric flux, so photolysis of molecules that absorb there (and O₃ chemistry, if any oxygen were present) is different. M dwarfs are also strongly active in the far-UV (including Lyman-α), which drives different photolysis channels.
- **Different shortwave absorption.** H₂O and CH₄ absorb strongly in the near-IR, where the M dwarf puts more of its energy, and Rayleigh scattering is weaker, so the same atmosphere can warm or cool differently.

The same caveats apply as in the first notebook: illustrative round-number fluxes, no organic haze, fixed CO₂ and N₂. The surface albedo is also held at 0.2 although it is expected to depend on the stellar spectrum.

**Runtime and convergence.** This notebook has not been run to convergence against the real GJ 876 spectrum by its author; expect to watch the first iterations. Weak near-UV output means less OH and slower CH₄ destruction than around the Sun, so CH₄ can build up to much larger abundances, the photochemical steady state can take considerably longer than in the Sun notebook (tens of minutes rather than minutes), and the coupling loop may need more than the default `max_iterations` or a looser `tolerance_K`. If CH₄/CO₂ becomes large, real atmospheres would form organic haze, which this model does not include, so treat the result with caution.

## 1. Environment

Photochem is distributed through **conda-forge** (it is not on PyPI). Starting from a bare Miniforge/conda install:

```sh
conda create -n photochem-env -c conda-forge python=3.11 photochem jupyter matplotlib
conda activate photochem-env
jupyter lab
```

or use the `environment.yml` next to this notebook: `conda env create -f environment.yml`.

The next cell checks that everything imports. If you launched Jupyter from a conda environment that lacks Photochem, you can also run `%conda install -c conda-forge photochem` in a cell and restart the kernel.

In [ ]:
import os
import time

import numpy as np
import yaml
import matplotlib.pyplot as plt

try:
    import photochem
    from photochem import EvoAtmosphere
    from photochem.clima import AdiabatClimate
    from photochem.utils import (
        stars,
        zahnle_rx_and_thermo_files,
        species_file_for_climate,
        settings_file_for_climate,
    )
except ImportError as err:
    raise ImportError(
        "Photochem is not installed in this kernel. See the install "
        "instructions in the cell above."
    ) from err

print("Photochem version:", photochem.__version__)

# All generated input files go here.
workdir = "archean_coupled_gj876"
os.makedirs(workdir, exist_ok=True)

## 2. Inputs

### Planet and star

Earth's mass and radius again. The reference instellation is the total flux the Sun delivered to Earth at 3 Ga, from Photochem's Sun model. `stars.muscles_spectrum` then downloads the GJ 876 spectrum from the MAST archive (**this step needs internet access**), appends a blackbody to extend it to long wavelengths, and rescales it to that instellation.

In [ ]:
planet_mass = 5.972e27      # g
planet_radius = 6.371e8     # cm
surface_albedo = 0.2
star_name = "GJ876"

# Instellation that the Sun at 3 Ga delivered to Earth (W/m^2)
wv_sun, flux_sun = stars.solar_spectrum(age=3.0)
instellation = stars.energy_in_spectrum(wv_sun, flux_sun)

# Observed GJ 876 spectrum, scaled to the same instellation
flux_file = f"{workdir}/{star_name}.txt"
wv, flux = stars.muscles_spectrum(star_name, outputfile=flux_file, stellar_flux=instellation)

# Orbital distance that corresponds to this instellation
L_sun, S_earth_now = 3.828e26, 1361.0
star = stars.MUSCLES_STARS[star_name]
luminosity = 4 * np.pi * (star["st_rad"] * 6.957e8) ** 2 * 5.670374e-8 * star["st_teff"] ** 4
distance_au = np.sqrt(luminosity / (4 * np.pi * instellation)) / 1.495978707e11
print(f"Instellation: {instellation:.0f} W/m^2; orbital distance ≈ {distance_au:.2f} AU; "
      f"L = {luminosity / L_sun:.4f} Lsun")

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(wv_sun, flux_sun, label="Sun at 3 Ga", alpha=0.8)
ax.loglog(wv, flux, label="GJ 876", alpha=0.8)
ax.set_xlim(100, 1e4); ax.set_ylim(1e-6, 1e4)
ax.set_xlabel("Wavelength (nm)"); ax.set_ylabel("Flux at planet (mW m$^{-2}$ nm$^{-1}$)")
ax.set_title("Same total instellation, different spectra")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

### Chemical network

A subset of Photochem's Zahnle reaction network containing H, He, N, O, C and S chemistry.

In [ ]:
mechanism_file = f"{workdir}/mechanism.yaml"
zahnle_rx_and_thermo_files(
    atoms_names=["H", "He", "N", "O", "C", "S"],
    rxns_filename=mechanism_file,
    thermo_filename=None,
    remove_reaction_particles=True,
)

### Boundary conditions (the "Archean" part)

Lower-boundary conditions set the atmosphere's sources and sinks:

- **N₂ = 1 bar, CO₂ = 0.01 bar** fixed partial pressures.
- **No O₂ source.** O₂ and O₃ stay at trace levels, unlike on modern Earth.
- **Volcanic H₂** and **volcanic CO, SO₂, H₂S, OCS** fluxes, and a **biological CH₄** flux. Fluxes are in molecules cm⁻² s⁻¹; `vdep` is a deposition velocity in cm s⁻¹.
- H₂ escapes to space at the diffusion limit.
- Water vapour follows a fixed **relative humidity of 50%** below the cold trap, and condenses above it.

Every gas not listed gets a zero-flux boundary by default.

In [ ]:
relative_humidity = 0.5

def bc(name, lower, upper=None):
    return {"name": name, "lower-boundary": lower,
            "upper-boundary": upper or {"type": "veff", "veff": 0.0}}

def surface_source(vdep, flux):
    # flux spread evenly over the lowest layer, plus deposition
    return {"type": "vdep + dist flux", "vdep": vdep, "flux": flux, "height": -1.0}

boundary_conditions = [
    # water: huge nominal pressure -> lower boundary is capped by saturation x RH
    bc("H2O", {"type": "press", "press": 270.0e6}),
    bc("N2",  {"type": "press", "press": 1.0e6}),       # 1 bar (dyn/cm^2)
    bc("CO2", {"type": "press", "press": 0.01e6}),      # 0.01 bar
    bc("O2",  {"type": "flux",  "flux": 0.0}),          # no oxygenic photosynthesis
    bc("H2",  surface_source(0.0, 3.0e10)),             # volcanic
    bc("CH4", {"type": "flux",  "flux": 3.0e10}),       # biological
    bc("CO",  surface_source(1.0e-8, 1.0e9)),
    bc("SO2", surface_source(1.0, 1.0e9)),
    bc("H2S", surface_source(0.015, 1.0e8)),
    bc("OCS", surface_source(0.003, 5.0e7)),
    bc("NO",  surface_source(0.01, 3.0e8)),
    bc("NO2", {"type": "vdep", "vdep": 0.05}),
    bc("NO3", {"type": "vdep", "vdep": 0.05}),
    bc("HNO3", {"type": "vdep", "vdep": 4.0}),
    bc("NH3", surface_source(1.0, 1.5e10)),
    bc("H2SO4", surface_source(1.0, 7.0e8)),
    bc("HCN", surface_source(0.13, 1.7e8)),
    bc("CH3CN", surface_source(0.13, 1.3e8)),
    bc("N2O", surface_source(0.0, 0.0)),
]

photochem_settings = {
    "atmosphere-grid": {"number-of-layers": 100},
    "planet": {
        "planet-mass": planet_mass,
        "planet-radius": planet_radius,
        "surface-albedo": surface_albedo,
        "solar-zenith-angle": 60.0,
        "hydrogen-escape": {"type": "diffusion limited"},
        "water": {"gas-rainout": True, "rainfall-rate": 1, "tropopause-altitude": 1.1e6},
    },
    "particles": [{"name": "H2Oaer", "RH-condensation": relative_humidity}],
    "boundary-conditions": boundary_conditions,
}
settings_file = f"{workdir}/photochem_settings.yaml"
with open(settings_file, "w") as f:
    yaml.safe_dump(photochem_settings, f, sort_keys=False)

### Initialize both models

The climate model tracks the radiatively important gases: H₂O and CO₂ (which can condense), plus N₂, CH₄ and H₂. Everything else in the photochemical network is chemically important but radiatively minor here.

In [ ]:
climate_species_file = f"{workdir}/climate_species.yaml"
climate_settings_file = f"{workdir}/climate_settings.yaml"

species_file_for_climate(
    climate_species_file,
    species=["H2O", "CO2", "N2", "CH4", "H2"],
    condensates=["H2O", "CO2"],
)
settings_file_for_climate(
    climate_settings_file, planet_mass, planet_radius, surface_albedo,
)

clim = AdiabatClimate(climate_species_file, climate_settings_file, flux_file)
clim.verbose = False
clim.RH = np.full(len(clim.species_names), relative_humidity)
clim.T_trop = 200.0   # initial guess for the stratospheric temperature (K)

pc = EvoAtmosphere(mechanism_file, settings_file, flux_file)
pc.var.verbose = 0

print("Climate species:", clim.species_names)

## 3. Coupled simulation

### Step 1: initial climate

With no photochemistry yet, guess trace CH₄ and H₂ and solve for the temperature structure. `surface_temperature` gives a fast approximate solution (adiabat + isothermal stratosphere), which we use as the starting point for a full radiative–convective equilibrium (`RCE`) solve.

In [ ]:
def climate_partial_pressures(mixing_ratios, total_pressure_bar):
    # Surface partial pressures (dyn/cm^2), ordered like clim.species_names.
    # H2O is given a large value; the climate model caps it at RH x saturation.
    out = np.full(len(clim.species_names), 1.0e-15)
    for i, name in enumerate(clim.species_names):
        if name == "H2O":
            out[i] = 270.0e6
        elif name in mixing_ratios:
            out[i] = mixing_ratios[name] * total_pressure_bar * 1.0e6
    return out

def solve_climate(P_surf_species, first_guess=False):
    if first_guess:
        clim.surface_temperature(P_surf_species, T_guess=280.0)   # fast approximate solve
    if not clim.RCE(P_surf_species, clim.T_surf, clim.T, clim.convecting_with_below):
        raise RuntimeError("RCE did not converge")

guess = {"N2": 0.98, "CO2": 0.01, "CH4": 1e-5, "H2": 1e-3}
solve_climate(climate_partial_pressures(guess, 1.0), first_guess=True)
T_surf_initial = clim.T_surf
print(f"Initial climate: T_surf = {clim.T_surf:.1f} K, tropopause at {clim.P_trop/1e6:.2f} bar")

def climate_profile():
    # Surface + layer centres, as a strictly decreasing pressure grid
    return np.append(clim.P_surf, clim.P), np.append(clim.T_surf, clim.T)

def eddy_diffusion(P_bar):
    # Kzz (cm^2/s): 1e5 in the troposphere, increasing with altitude above.
    return np.where(P_bar > 0.1, 1.0e5, np.minimum(1.0e5 * (0.1 / P_bar) ** 0.5, 1.0e7))

### Step 2: give the climate to the photochemical model

`initialize_atmosphere_p` takes a pressure-based *P*–*T*–*K*<sub>zz</sub> profile. With `persistent=True` the profile stays fixed in pressure space while composition evolves, and is updated in place later with `set_press_temp_edd_profile`.

In [ ]:
P, T = climate_profile()
pc.initialize_atmosphere_p(
    P, T, eddy_diffusion(P / 1e6),
    mix=None,
    particle_radius={"H2Oaer": np.full(P.size, 1.0e-3)},   # 10 micron cloud drops
    persistent=True,
    tropopause_pressure=float(clim.P_trop),
    target_pressure=P[-1],
)
P_init, T_init = P.copy(), T.copy()

### Step 3: iterate to a self-consistent solution

Each pass runs the photochemistry to steady state, hands the surface CH₄, H₂, CO₂ and N₂ abundances to the climate model, and solves for the new temperature profile. The first photochemical solve starts from scratch and takes a few minutes; later solves restart from the previous state and are much faster.

In [ ]:
max_iterations = 10
tolerance_K = 0.1

history = []
start = time.perf_counter()
for iteration in range(max_iterations):
    # photochemistry on the current T(P)
    converged = pc.find_steady_state()
    if not converged:
        print("Warning: photochemistry did not fully converge this iteration")
    atm = pc.mole_fraction_dict()
    total_bar = pc.wrk.surface_pressure

    # climate on the photochemical composition
    mixing = {sp: atm[sp][0] for sp in ["N2", "CO2", "CH4", "H2"]}
    T_previous = clim.T_surf
    solve_climate(climate_partial_pressures(mixing, total_bar))

    history.append({"iteration": iteration, "T_surf": clim.T_surf,
                    "CH4": mixing["CH4"], "H2": mixing["H2"]})
    print(f"iteration {iteration}: T_surf = {clim.T_surf:7.3f} K "
          f"(change {clim.T_surf - T_previous:+.3f} K), "
          f"CH4 = {mixing['CH4']:.2e}, H2 = {mixing['H2']:.2e}, "
          f"elapsed {time.perf_counter() - start:.0f} s")

    if iteration > 0 and abs(clim.T_surf - T_previous) < tolerance_K:
        print("Converged.")
        break

    # pass the new T(P) back to the photochemical model
    pc.destroy_stepper()
    P, T = climate_profile()
    pc.set_press_temp_edd_profile(
        P, T, eddy_diffusion(P / 1e6),
        trop_p=float(clim.P_trop),
        hydro_pressure=True,
        maintain_toa_pressure=True,
        target_pressure=P[-1],
    )
else:
    print("Reached max_iterations without meeting the tolerance.")

## 4. Results

In [ ]:
atm = pc.mole_fraction_dict()
P_final, T_final = climate_profile()

print(f"Surface temperature: {clim.T_surf:.1f} K ({clim.T_surf - 273.15:.1f} °C)")
print(f"Surface pressure:    {pc.wrk.surface_pressure:.3f} bar")
print(f"Tropopause pressure: {clim.P_trop/1e6:.3f} bar")
print()
print("Surface mixing ratios:")
for sp in ["N2", "CO2", "H2O", "H2", "CH4", "CO", "C2H6", "HCN", "O2", "O3"]:
    print(f"  {sp:5s} {atm[sp][0]:.2e}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))

# Temperature structure
ax = axes[0]
ax.plot(T_init, P_init / 1e6, "--", color="C7", label="initial (trace CH$_4$, H$_2$)")
ax.plot(T_final, P_final / 1e6, color="C3", lw=2, label="coupled solution")
ax.set_yscale("log"); ax.invert_yaxis()
ax.set_xlabel("Temperature (K)"); ax.set_ylabel("Pressure (bar)")
ax.set_title("Temperature"); ax.legend(); ax.grid(alpha=0.3)

# Composition
ax = axes[1]
for sp in ["N2", "CO2", "H2O", "H2", "CH4", "CO", "C2H6", "HCN"]:
    ax.plot(atm[sp], atm["pressure"] / 1e6, label=sp)
ax.set_xscale("log"); ax.set_yscale("log"); ax.invert_yaxis()
ax.set_xlim(1e-12, 2)
ax.set_xlabel("Mixing ratio"); ax.set_title("Composition")
ax.legend(ncol=2, fontsize=8); ax.grid(alpha=0.3)

# Convergence
ax = axes[2]
ax.plot(range(len(history) + 1), [T_surf_initial] + [h["T_surf"] for h in history], "o-")
ax.ticklabel_format(useOffset=False, axis="y")
ax.set_xlabel("Coupling iteration (0 = initial guess)"); ax.set_ylabel("Surface temperature (K)")
ax.set_title("Convergence"); ax.grid(alpha=0.3)
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

plt.tight_layout(); plt.show()

### Diagnostics

Photochem can decompose the budget of any species. Here, what makes and destroys methane at steady state:

In [ ]:
budget = pc.production_and_loss("CH4", pc.wrk.usol)
print("Column-integrated CH4 production (molecules cm^-2 s^-1):")
for name, rate in list(zip(budget.production_rx, budget.integrated_production))[:4]:
    print(f"  {name:30s} {rate:.2e}")
print("Column-integrated CH4 loss (molecules cm^-2 s^-1):")
for name, rate in list(zip(budget.loss_rx, budget.integrated_loss))[:4]:
    print(f"  {name:30s} {rate:.2e}")

surface_fluxes, top_fluxes = pc.gas_fluxes()
print(f"\nH2 escape flux at top of atmosphere: {top_fluxes['H2']:.2e} molecules cm^-2 s^-1")

## 5. Comparison with the Sun at 3 Ga

Reference values below are the converged results of the first notebook (same planet, same instellation, Sun at 3 Ga spectrum). Re-run that notebook if you change any shared setting.

In [ ]:
sun_3Ga = {"T_surf (K)": 284.1, "surface pressure (bar)": 1.060,
           "H2O": 5.50e-3, "H2": 5.68e-3, "CH4": 4.22e-5, "CO": 9.45e-5, "C2H6": 1.42e-9, "HCN": 4.98e-11}
this_run = {"T_surf (K)": clim.T_surf, "surface pressure (bar)": pc.wrk.surface_pressure}
this_run.update({sp: atm[sp][0] for sp in ["H2O", "H2", "CH4", "CO", "C2H6", "HCN"]})

print(f"{'':26s}{'Sun 3 Ga':>12s}{star_name:>12s}")
for key in sun_3Ga:
    fmt = "{:12.3f}" if ("(" in key) else "{:12.2e}"
    print(f"{key:26s}" + fmt.format(sun_3Ga[key]) + fmt.format(this_run[key]))